# Projeto: Inserção dos dados na camada Gold do banco de dados.

### Descrição Geral

Este notebook documenta o processo de identificação de **municípios em risco de alagamento**, com base na análise comparativa entre:
- A tabela `clima` da camada **gold**, contendo previsões meteorológicas padronizadas por município;
- O arquivo `chuvas_criticas`, proveniente da camada **silver**, validado por regras automatizadas via **Great Expectations (GX)** e contendo o volume máximo de chuva suportado por município.

O objetivo central é gerar uma lista consolidada de localidades em que a **precipitação prevista excede o volume crítico suportado**, caracterizando um cenário de **potencial alagamento**. Os dados resultantes são enriquecidos e persistidos na tabela `alertas_alagamentos` da camada **gold**, servindo como base para geração de alertas, ações preventivas e análises de risco climático.

Esse cruzamento crítico transforma previsões climáticas brutas em **informação acionável**, com alto valor analítico para políticas públicas, defesa civil e monitoramento ambiental.


### Informações Técnicas

- **Fonte de Dados**:
  * Tabela da base gold referentes a previsões climáticas
    * `clima`
  * Tabela Silver de Chuvas Críticas
    * `silver_chuvas_criticas`
- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks

- **Dados persistidos**:
  - Data de carga
  - Data da previsão do clima
  - Nome do município
  - Nome do estado
  - País
  - Volume máximo de chuva
  - Precipitação prevista
  - Propabilidade da previsão prevista
  - Fonte da previsão do tempo
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 20/05/2025 | Lucas Berta e Lucas Bringmann      |  1.0   | Criação da inserção dos dados para camada gold|
| 20/05/2025 | Gabriel Rost      |  1.1   | Inclusão de documentação sobre enrequecimentos realizados|
| 25/05/2025 | Evelise Dias      |  1.2  | 	Adição da leitura de arquivos validados pelo Great Expectations|
| 26/05/2025 | Evelise Dias      |  1.3  | 	Atualização da descrição geral do projeto e alinhamento com a arquitetura atual|

In [0]:
%run ../../src/utils/utils_run_gold

In [0]:
%run ./queries_gold

In [0]:
df_chuva_path = lakehouse_logs_gx + "/silver/chuvas_criticas/validos/chuvas_criticas"
df_chuva = ler_tabelas(df_chuva_path, fromTable=False)

In [0]:
df_clima = ler_tabelas(caminho_tabela_clima_gold)

In [0]:
df_chuva = df_chuva.withColumnRenamed("sig_uf", "nom_estado")

In [0]:
df = (
    df_clima.alias("clima")
    .join(df_chuva.alias("chuva"), on="nom_municipio", how="inner")
    .select(
        col("chuva.vol_max_chuva"),
        col("clima.nom_municipio"),
        col("clima.nom_estado"),
        col("clima.dt_carga"),
        col("clima.dt_previsao"),
        col("clima.precipitacao").alias("precipitacao_prevista"),
        col("clima.prob_precipitacao").alias("prob_precipitacao_prevista"),
        col("clima.flg_fonte_previsao"),
        col("clima.id"),
        col("chuva.pais")
    )
    .where(col("clima.precipitacao") > col("chuva.vol_max_chuva"))
)

In [0]:
spark.sql(create_table_gold_alertas_alagamentos)

In [0]:
inserir_com_merge(df, caminho_tabela_alertas_alagamentos_gold, ["id"])